<a href="https://colab.research.google.com/github/KooZiChen/Deep-Reinforcement-Learning/blob/main/unit_06/unit6_a2c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 6 - A2C with Panda Robotics Environments

Train an A2C (Advantage Actor-Critic) agent on two robotics environments from `panda_gym` using Stable-Baselines3, then push the trained models to the Hugging Face Hub.

**Environments:**
- `PandaReachDense-v3` — robot arm reaches a target position
- `PandaPickAndPlace-v3` — robot arm picks up and places an object

**Instructions:**
1. Set runtime to GPU: Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Log in to Hugging Face when prompted

In [ ]:
# Install system display libraries (needed for rendering in Colab)
!apt-get install -y xvfb python3-opengl ffmpeg > /dev/null 2>&1

# Install Python dependencies
!pip install stable-baselines3[extra] gymnasium panda_gym huggingface_sb3 huggingface_hub pyvirtualdisplay -q

In [ ]:
# Set up virtual display for headless rendering
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("Virtual display started.")

In [ ]:
import os
import gymnasium as gym
import panda_gym

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.evaluation import evaluate_policy

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

print("Imports successful.")

In [ ]:
# Log in to Hugging Face Hub
notebook_login()

# Store credentials for git operations
!git config --global credential.helper store

---
## Environment 1: PandaReachDense-v3

The robot arm must reach a target position. Dense rewards are given based on distance to the goal.

In [ ]:
# Explore observation and action spaces
env = gym.make("PandaReachDense-v3")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("\nSample observation:")
obs, _ = env.reset()
for key, val in obs.items():
    print(f"  {key}: shape={val.shape}, dtype={val.dtype}")

env.close()

In [ ]:
# Create vectorized + normalized environment
env_reach = make_vec_env(
    "PandaReachDense-v3",
    n_envs=4,
    seed=0
)

env_reach = VecNormalize(
    env_reach,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.0
)

print("PandaReachDense-v3 vectorized env ready (4 envs, normalized).")

In [ ]:
# Define and train A2C model
model_reach = A2C(
    policy="MultiInputPolicy",
    env=env_reach,
    verbose=1,
    device="auto"
)

model_reach.learn(total_timesteps=1_000_000)
print("Training complete for PandaReachDense-v3.")

In [ ]:
# Save model and VecNormalize stats
model_reach.save("a2c-PandaReachDense-v3")
env_reach.save("vec_normalize_reach.pkl")
print("Model and VecNormalize stats saved.")

In [ ]:
# Evaluate the trained agent
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

eval_env_reach = make_vec_env("PandaReachDense-v3", n_envs=1, seed=42)
eval_env_reach = VecNormalize.load("vec_normalize_reach.pkl", eval_env_reach)
eval_env_reach.training = False
eval_env_reach.norm_reward = False

mean_reward, std_reward = evaluate_policy(
    model_reach,
    eval_env_reach,
    n_eval_episodes=10,
    deterministic=True
)
print(f"PandaReachDense-v3 — Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
eval_env_reach.close()

In [ ]:
# Push to Hugging Face Hub
package_to_hub(
    model=model_reach,
    model_name="a2c-PandaReachDense-v3",
    model_architecture="A2C",
    env_id="PandaReachDense-v3",
    eval_env=eval_env_reach,
    repo_id="TheBestMoldyCheese/a2c-PandaReachDense-v3",
    commit_message="Upload A2C agent trained on PandaReachDense-v3 - Unit 6 HF DRL Course"
)

---
## Environment 2: PandaPickAndPlace-v3

The robot arm must pick up an object and place it at a target location. This is a harder task requiring manipulation skills.

In [ ]:
# Create vectorized + normalized environment
env_pnp = make_vec_env(
    "PandaPickAndPlace-v3",
    n_envs=4,
    seed=0
)

env_pnp = VecNormalize(
    env_pnp,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.0
)

print("PandaPickAndPlace-v3 vectorized env ready (4 envs, normalized).")

In [ ]:
# Define and train A2C model
model_pnp = A2C(
    policy="MultiInputPolicy",
    env=env_pnp,
    verbose=1,
    device="auto"
)

model_pnp.learn(total_timesteps=1_000_000)
print("Training complete for PandaPickAndPlace-v3.")

In [ ]:
# Save model and VecNormalize stats
model_pnp.save("a2c-PandaPickAndPlace-v3")
env_pnp.save("vec_normalize_pnp.pkl")
print("Model and VecNormalize stats saved.")

In [ ]:
# Evaluate the trained agent
eval_env_pnp = make_vec_env("PandaPickAndPlace-v3", n_envs=1, seed=42)
eval_env_pnp = VecNormalize.load("vec_normalize_pnp.pkl", eval_env_pnp)
eval_env_pnp.training = False
eval_env_pnp.norm_reward = False

mean_reward, std_reward = evaluate_policy(
    model_pnp,
    eval_env_pnp,
    n_eval_episodes=10,
    deterministic=True
)
print(f"PandaPickAndPlace-v3 — Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
eval_env_pnp.close()

In [ ]:
# Push to Hugging Face Hub
package_to_hub(
    model=model_pnp,
    model_name="a2c-PandaPickAndPlace-v3",
    model_architecture="A2C",
    env_id="PandaPickAndPlace-v3",
    eval_env=eval_env_pnp,
    repo_id="TheBestMoldyCheese/a2c-PandaPickAndPlace-v3",
    commit_message="Upload A2C agent trained on PandaPickAndPlace-v3 - Unit 6 HF DRL Course"
)